# SkillGap - Phase 1: Exploratory Data Analysis (EDA)

This notebook inspects the raw datasets required for SkillGap:
- **Job Postings** (`data/raw/job_postings.csv`): Role titles and descriptions to profile skill demand.
- **Resumes** (`data/raw/resumes.csv`): Candidate profiles categorized by domain.
- **ESCO Skills** (`data/raw/esco_skills.csv`): Official European Commission skill taxonomy.
- **Courses** (`data/raw/courses.csv`): Course catalog for gap remediation.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is on sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from config import settings
from ml.data_io import (
    load_esco_skills,
    load_job_postings,
    load_resumes,
    load_courses,
    DatasetNotFoundError
)

print(f"Project root: {ROOT_DIR}")
print(f"Raw data directory: {settings.RAW_DATA_DIR}")

## 1. Dataset Availability & Row Counts
We load each dataset via the unified `ml.data_io` loaders that handle whitespace cleaning, column mapping, and deduplication.

In [2]:
datasets = {}
status_rows = []

# 1. Job Postings
try:
    df_jobs = load_job_postings()
    datasets["job_postings"] = df_jobs
    status_rows.append({"Dataset": "job_postings.csv", "Status": "Loaded", "Rows": len(df_jobs), "Columns": list(df_jobs.columns)})
except DatasetNotFoundError:
    status_rows.append({"Dataset": "job_postings.csv", "Status": "Missing (run scripts/check_data.py)", "Rows": 0, "Columns": []})

# 2. Resumes
try:
    df_resumes = load_resumes()
    datasets["resumes"] = df_resumes
    status_rows.append({"Dataset": "resumes.csv", "Status": "Loaded", "Rows": len(df_resumes), "Columns": list(df_resumes.columns)})
except DatasetNotFoundError:
    status_rows.append({"Dataset": "resumes.csv", "Status": "Missing (run scripts/check_data.py)", "Rows": 0, "Columns": []})

# 3. ESCO Skills
try:
    df_esco = load_esco_skills()
    datasets["esco_skills"] = df_esco
    status_rows.append({"Dataset": "esco_skills.csv", "Status": "Loaded", "Rows": len(df_esco), "Columns": list(df_esco.columns)})
except DatasetNotFoundError:
    status_rows.append({"Dataset": "esco_skills.csv", "Status": "Missing (run scripts/check_data.py)", "Rows": 0, "Columns": []})

# 4. Courses
try:
    df_courses = load_courses()
    datasets["courses"] = df_courses
    status_rows.append({"Dataset": "courses.csv", "Status": "Loaded", "Rows": len(df_courses), "Columns": list(df_courses.columns)})
except DatasetNotFoundError:
    status_rows.append({"Dataset": "courses.csv", "Status": "Missing (run scripts/check_data.py)", "Rows": 0, "Columns": []})

summary_df = pd.DataFrame(status_rows)
summary_df

## 2. Job Postings Analysis: Top 20 Job Titles
If the real `job_postings.csv` has been placed in `data/raw/`, we compute the top 20 job titles by frequency.

In [3]:
if "job_postings" in datasets:
    df_jobs = datasets["job_postings"]
    print("Top 20 Job Titles in Postings:")
    top_20 = df_jobs["title"].value_counts().head(20)
    display(top_20)
else:
    print("job_postings.csv is not present in data/raw/.")
    print("To populate this analysis, download the dataset as instructed by scripts/check_data.py and place it in data/raw/job_postings.csv.")

## 3. Sample Job Postings (5 Examples)
Inspecting 5 actual job descriptions and metadata to examine skill terminology.

In [4]:
if "job_postings" in datasets:
    df_jobs = datasets["job_postings"]
    sample_cols = [c for c in ["title", "company", "location", "description"] if c in df_jobs.columns]
    sample_df = df_jobs[sample_cols].head(5).copy()
    sample_df["description_snippet"] = sample_df["description"].str.slice(0, 150) + "..."
    display(sample_df.drop(columns=["description"] if "description" in sample_df.columns else []))
else:
    print("job_postings.csv is missing. Skipping sample display.")

## 4. Resume Dataset Analysis: Category Distribution
Inspecting the domain distribution of labeled candidate resumes.

In [5]:
if "resumes" in datasets:
    df_resumes = datasets["resumes"]
    print("Resume Category Counts:")
    category_counts = df_resumes["category"].value_counts()
    display(category_counts)
else:
    print("resumes.csv is not present in data/raw/.")
    print("To populate this analysis, place your resume dataset in data/raw/resumes.csv.")